In [2]:
import pandas as pd

df = pd.read_excel('Dataset for Data Analytics.xlsx')

print(df.shape)
print(df.dtypes)
df.head()

(1200, 14)
OrderID                    object
Date               datetime64[ns]
CustomerID                 object
Product                    object
Quantity                    int64
UnitPrice                 float64
ShippingAddress            object
PaymentMethod              object
OrderStatus                object
TrackingNumber             object
ItemsInCart                 int64
CouponCode                 object
ReferralSource             object
TotalPrice                float64
dtype: object


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [3]:
print(df.isna().sum())

OrderID              0
Date                 0
CustomerID           0
Product              0
Quantity             0
UnitPrice            0
ShippingAddress      0
PaymentMethod        0
OrderStatus          0
TrackingNumber       0
ItemsInCart          0
CouponCode         309
ReferralSource       0
TotalPrice           0
dtype: int64


In [5]:
df['CouponCode'] = df['CouponCode'].fillna('NoCoupon')

print(df['CouponCode'].isna().sum())
print(df['CouponCode'].value_counts())

0
CouponCode
FREESHIP    313
NoCoupon    309
WINTER15    292
SAVE10      286
Name: count, dtype: int64


In [6]:
Q1 = df['TotalPrice'].quantile(0.25)
Q3 = df['TotalPrice'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Normal range:", lower_bound, "to", upper_bound)

outliers = df[(df['TotalPrice'] < lower_bound) | (df['TotalPrice'] > upper_bound)]
print("Number of outliers:", len(outliers))
outliers[['OrderID', 'Quantity', 'UnitPrice', 'TotalPrice']]

Normal range: -1341.4125 to 3330.4075
Number of outliers: 8


,OrderID,Quantity,UnitPrice,TotalPrice
107,ORD200107,5,670.75,3353.75
326,ORD200326,5,670.48,3352.40
328,ORD200328,5,674.04,3370.20
469,ORD200469,5,676.98,3384.90
632,ORD200632,5,678.16,3390.80
789,ORD200789,5,691.28,3456.40
1065,ORD201065,5,666.80,3334.00
1122,ORD201122,5,678.19,3390.95


In [7]:
df['TotalPrice'] = df['TotalPrice'].clip(lower=lower_bound, upper=upper_bound)
outliers_after = df[(df['TotalPrice'] < lower_bound) | (df['TotalPrice'] > upper_bound)]
print("Outliers remaining:", len(outliers_after))

Outliers remaining: 0


In [9]:
df['OrderMonth'] = df['Date'].dt.month
df['HasDiscount'] = df['CouponCode'] != 'NoCoupon'
df['CartConversionRate'] = df['Quantity'] / df['ItemsInCart']

In [10]:
df[['Date', 'OrderMonth', 'CouponCode', 'HasDiscount', 'Quantity', 'ItemsInCart', 'CartConversionRate']].head()

,Date,OrderMonth,CouponCode,HasDiscount,Quantity,ItemsInCart,CartConversionRate
0,2023-01-04,1,SAVE10,True,5,7,0.714286
1,2024-08-23,8,SAVE10,True,2,3,0.666667
2,2024-02-27,2,FREESHIP,True,5,8,0.625000
3,2023-10-15,10,SAVE10,True,1,5,0.200000
4,2025-05-08,5,SAVE10,True,4,8,0.500000


In [11]:
df.to_excel('cleaned_data.xlsx', index=False)